In [ ]:
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("/home/h2048/data/py/0112/scarches_mapping_output/query_mapped_to_reference.h5ad")  # 你的 step2 输出
epi = adata.obs["cell_type_mapped"].astype(str).eq("Epithelial")

tbl = (adata.obs.loc[epi]
       .groupby("dataset")[["mapping_confidence","mapping_margin"]]
       .agg(n=("mapping_confidence","size"),
            conf_median=("mapping_confidence","median"),
            margin_median=("mapping_margin","median"))
       .sort_values(["conf_median","margin_median"]))

print(tbl.head(10))   # 最差的几个 dataset
print(tbl.tail(10))   # 最好的几个 dataset


In [ ]:
import numpy as np
import pandas as pd

obs = adata.obs.copy()
epi = obs["cell_type_mapped"].astype(str).eq("Epithelial")
x = obs.loc[epi, ["dataset","mapping_confidence","mapping_margin"]].copy()

g = x.groupby("dataset", observed=True)

df = pd.DataFrame({
    "n": g.size(),
    "conf_q05": g["mapping_confidence"].quantile(0.05),
    "conf_median": g["mapping_confidence"].median(),
    "conf_q95": g["mapping_confidence"].quantile(0.95),
    "frac_conf_lt_0p8": g["mapping_confidence"].apply(lambda s: (s < 0.8).mean()),
    "frac_conf_lt_0p9": g["mapping_confidence"].apply(lambda s: (s < 0.9).mean()),
    "margin_median": g["mapping_margin"].median(),
    "frac_margin_lt_0p5": g["mapping_margin"].apply(lambda s: (s < 0.5).mean()),
}).sort_values(["frac_conf_lt_0p8","conf_median"], ascending=[False, True])

print(df.head(15))


In [ ]:
import pandas as pd

qc_cols = [c for c in ["total_counts","n_genes_by_counts","pct_counts_mt","pct_counts_ribo"] if c in adata.obs.columns]
obs = adata.obs.copy()
epi = obs["cell_type_mapped"].astype(str).eq("Epithelial")

for c in qc_cols:
    tmp = obs.loc[epi, [c, "mapping_confidence"]].dropna()
    if len(tmp) > 0:
        print(c, tmp.corr(method="spearman").iloc[0,1])


In [ ]:
import scanpy as sc
genes = [g for g in ["EPCAM","KRT19","PTPRC","LYZ"] if g in adata.var_names]
sc.tl.score_genes(adata, gene_list=[g for g in ["EPCAM","KRT19"] if g in adata.var_names], score_name="epi_score")
sc.tl.score_genes(adata, gene_list=[g for g in ["PTPRC","LYZ"] if g in adata.var_names], score_name="immune_score")

obs = adata.obs
epi = obs["cell_type_mapped"].astype(str).eq("Epithelial")
sus = epi & (obs["immune_score"] > 0.2)  # 阈值你可按分布调
print("suspected epithelial-immune doublet fraction:", sus.mean())
print(obs.loc[sus, ["dataset","mapping_confidence"]].groupby("dataset", observed=True).size().sort_values(ascending=False).head(10))


In [ ]:
import numpy as np
import scanpy as sc

key = "X_scANVI_mapped"
adata.obsm["X_latent"] = adata.obsm[key]

epi_mask = adata.obs["cell_type_mapped"].astype(str).eq("Epithelial")
adata_epi = adata[epi_mask].copy()

sc.pp.neighbors(adata_epi, use_rep="X_latent", n_neighbors=30)

# 计算每个细胞：邻居中同 label 的比例（这里先用 dataset 或 cell_type_mapped 都行）
label = adata_epi.obs["dataset"].astype(str)   # 你也可以换成更有意义的上皮细分标签
conn = adata_epi.obsp["connectivities"].tocsr()

purity = np.zeros(adata_epi.n_obs, dtype=float)
for i in range(adata_epi.n_obs):
    nbr = conn[i].indices
    purity[i] = (label.iloc[nbr].values == label.iloc[i]).mean() if len(nbr) else np.nan

adata_epi.obs["knn_same_dataset_frac"] = purity
print(adata_epi.obs.groupby("dataset", observed=True)["knn_same_dataset_frac"].median().sort_values(ascending=False).head(10))


In [ ]:
from sklearn.metrics import silhouette_score

X = adata_epi.obsm["X_latent"]
labels = adata_epi.obs["dataset"].astype(str).values
# 数据量太大可以先抽样
import numpy as np
idx = np.random.RandomState(0).choice(adata_epi.n_obs, size=min(50000, adata_epi.n_obs), replace=False)
print("silhouette(dataset) on epithelial latent:", silhouette_score(X[idx], labels[idx]))


In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd

# 1) 在全体细胞上建邻接图（用 mapped latent）
adata.obsm["X_latent"] = adata.obsm["X_scANVI_mapped"]
sc.pp.neighbors(adata, use_rep="X_latent", n_neighbors=30)

# 2) 计算每个细胞：邻居中同为 Epithelial 的比例
conn = adata.obsp["connectivities"].tocsr()
ctype = adata.obs["cell_type_mapped"].astype(str).values
is_epi = (ctype == "Epithelial")

epi_neighbor_frac = np.zeros(adata.n_obs, dtype=float)
for i in range(adata.n_obs):
    nbr = conn[i].indices
    epi_neighbor_frac[i] = is_epi[nbr].mean() if len(nbr) else np.nan

adata.obs["epi_neighbor_frac"] = epi_neighbor_frac

# 3) 只看上皮，并按 dataset 汇总
obs = adata.obs
m = obs["cell_type_mapped"].astype(str).eq("Epithelial")
tbl = (obs.loc[m]
       .groupby("dataset", observed=True)["epi_neighbor_frac"]
       .agg(n="size", median="median", q05=lambda s: s.quantile(0.05))
       .sort_values("median"))
print(tbl.head(15))


In [ ]:
print([k for k in adata.obsm_keys() if "prob" in k.lower() or "proba" in k.lower() or "pred" in k.lower()])
import numpy as np

probs = adata.obsm["pred_probs"]   # 改成你的实际 key
classes = np.array(["B","Endothelial","Epithelial","Fibroblast","Myeloid","SMC","T"])  # 按你模型顺序修正

top1 = probs.argmax(1)
tmp = probs.copy()
tmp[np.arange(tmp.shape[0]), top1] = -1
top2 = tmp.argmax(1)

adata.obs["top2_label"] = classes[top2]


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

genes = ["SFTPC","SFTPA1","SFTPA2","ABCA3","SLC34A2"]
genes = [g for g in genes if g in adata.var_names]

epi = adata.obs["cell_type_mapped"].astype(str).eq("Epithelial")
low = epi & (adata.obs["mapping_confidence"] < 0.7)

# 每个基因在低置信上皮里的阳性比例（>0）
X = adata[low, genes].X
if not hasattr(X, "toarray"):
    arr = np.asarray(X)
else:
    arr = X.toarray()

pos = (arr > 0).mean(axis=0)
print(pd.Series(pos, index=genes).sort_values(ascending=False))


In [ ]:
import numpy as np
import scanpy as sc

adata.obsm["X_latent"] = adata.obsm["X_scANVI_mapped"]
sc.pp.neighbors(adata, use_rep="X_latent", n_neighbors=30)

conn = adata.obsp["connectivities"].tocsr()
ctype = adata.obs["cell_type_mapped"].astype(str).values
is_epi = (ctype == "Epithelial")

frac = np.zeros(adata.n_obs, float)
for i in range(adata.n_obs):
    nbr = conn[i].indices
    frac[i] = is_epi[nbr].mean() if len(nbr) else np.nan
adata.obs["epi_neighbor_frac"] = frac

epi = adata.obs["cell_type_mapped"].astype(str).eq("Epithelial")
print(adata.obs.loc[epi].groupby("dataset", observed=True)["epi_neighbor_frac"].median().sort_values().head(10))


In [ ]:
import numpy as np
import pandas as pd

conn = adata.obsp["connectivities"].tocsr()
ctype = adata.obs["cell_type_mapped"].astype(str).values
ds = adata.obs["dataset"].astype(str).values

is_epi = (ctype == "Epithelial")
epi_idx = np.where(is_epi & (ds == "GSE299751"))[0]

rows = []
for i in epi_idx:
    nbr = conn[i].indices
    nbr_types = ctype[nbr]
    non = nbr_types[nbr_types != "Epithelial"]
    if len(non) == 0:
        rows.append("None")
    else:
        rows.append(pd.Series(non).value_counts().idxmax())

print(pd.Series(rows).value_counts())


In [ ]:
obs = adata.obs
epi = obs["cell_type_mapped"].astype(str).eq("Epithelial")

obs["epi_tier"] = "non_epi"
obs.loc[epi & (obs["epi_neighbor_frac"] >= 0.90), "epi_tier"] = "A"
obs.loc[epi & (obs["epi_neighbor_frac"] < 0.80),  "epi_tier"] = "C"
obs.loc[epi & (obs["epi_tier"].eq("non_epi")),   "epi_tier"] = "B"  # 剩余上皮

print(obs.loc[epi].groupby(["dataset","epi_tier"], observed=True).size().unstack(fill_value=0).sort_values("C", ascending=False).head(15))


In [ ]:
import scanpy as sc

# 免疫/间质/内皮简单 signature
sig = {
  "Immune": ["PTPRC","LST1","LYZ","TYROBP"],
  "Fibro": ["COL1A1","DCN","LUM"],
  "Endo": ["PECAM1","VWF","KDR"],
  "SMC": ["ACTA2","TAGLN","MYH11"],
  "Epi": ["EPCAM","KRT19","KRT8","KRT18"]
}
for name, genes in sig.items():
    genes = [g for g in genes if g in adata.var_names]
    if len(genes) >= 3:
        sc.tl.score_genes(adata, genes, score_name=f"sig_{name}")

# 看 Tier C 的中位数特征
c = obs["epi_tier"].eq("C")
cols = [c for c in obs.columns if c.startswith("sig_")]
print(obs.loc[c, cols].median().sort_values(ascending=False))
print(obs.loc[c].groupby("dataset", observed=True)[cols].median().sort_values("sig_Immune", ascending=False).head(10))


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

obs = adata.obs
epi = obs["cell_type_mapped"].astype(str).eq("Epithelial")
C = epi & obs["epi_tier"].eq("C")

sig = {
  "Epi":  ["EPCAM","KRT19","KRT8","KRT18","KRT17"],
  "Fibro":["COL1A1","COL1A2","DCN","LUM","COL3A1"],
  "Imm":  ["PTPRC","LST1","LYZ","TYROBP","FCER1G"],
  "Endo":["PECAM1","VWF","KDR","EMCN"],
  "SMC": ["ACTA2","TAGLN","MYH11","CNN1"]
}

for name, genes in sig.items():
    genes = [g for g in genes if g in adata.var_names]
    if len(genes) >= 3:
        sc.tl.score_genes(adata, genes, score_name=f"sig_{name}")

cols = [f"sig_{k}" for k in sig.keys() if f"sig_{k}" in obs.columns]

# 规则：Fibro/Imm 明显高于 Epi 的，优先改判或 Unknown
obs["C_subtype"] = "non_C"
obs.loc[C, "C_subtype"] = "C2_epi_candidate"

fibro_like = C & (obs["sig_Fibro"] > obs["sig_Epi"] + 0.2)
imm_like   = C & (obs["sig_Imm"]   > obs["sig_Epi"] + 0.2)
endo_like  = C & (("sig_Endo" in obs.columns) & (obs["sig_Endo"] > obs["sig_Epi"] + 0.2))

obs.loc[fibro_like, "C_subtype"] = "C1_fibro_like"
obs.loc[imm_like,   "C_subtype"] = "C1_immune_like"
obs.loc[endo_like,  "C_subtype"] = "C1_endo_like"

print(obs.loc[C].groupby(["dataset","C_subtype"], observed=True).size().unstack(fill_value=0).sort_values("C1_fibro_like", ascending=False).head(15))


In [ ]:
import celltypist

# 加载你的模型
model = celltypist.models.Model.load(model="/home/h2048/data/source/reference/celltypist_models/Cells_Lung_Airway.pkl")

# 获取所有细胞类型
cell_types = model.classifier.classes_

print(f"Total cell types: {len(cell_types)}")
print("\nAll cell types:")
for i, ct in enumerate(cell_types):
    print(f"{i+1}. {ct}")